- 타이타닉 데이터셋 입력 컬럼(Feature) 상세 설명

| 컬럼명 | 설명 및 의미 | 비고 / 값 예시 |
| pclass | 객실 등급 (Ticket class) | 1 = 1등석, 2 = 2등석, 3 = 3등석 |
| sex | 성별 | male (남성), female (여성) |
| age | 나이 | 결측치 존재, 1세 미만은 소수점 표시 |
| sibsp | 동승한 형제/자매/배우자의 수 | 0, 1, 2 ... |
| parch | 동승한 부모/자녀의 수 | 0, 1, 2 ... |
| fare | 탑승 요금 | 승객이 지불한 운임 (파운드화) |
| embarked | 탑승 항구 코드 | C = Cherbourg, Q = Queenstown, S = Southampton |
| class | 객실 등급 텍스트 | First, Second, Third (pclass와 동일 정보) |
| who | 승객 구분 | man (성인 남성), woman (성인 여성), child (아이) |
| deck | 선실 구역 번호 | A, B, C, D, E, F, G (결측치 매우 많음) |
| embark_town | 탑승 도시 이름 | Cherbourg, Queenstown, Southampton |
| alone | 혼자 탑승했는지 여부 | True (동승자 없음), False (동승자 있음) |

In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns 
import platform
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression, LinearRegression, Lasso, Ridge
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay, mean_squared_error, root_mean_squared_error

# sklearn.set_config(display="text")

# 1. 운영체제별 한글 폰트 설정
if platform.system() == 'Windows':
    plt.rc('font', family='Malgun Gothic')
elif platform.system() == 'Darwin': # Mac OS
    plt.rc('font', family='AppleGothic')
else: # Linux (Colab 등)
    plt.rc('font', family='NanumBarunGothic')

 
# 2. 마이너스 기호(-) 깨짐 방지
plt.rcParams['axes.unicode_minus'] = False

In [4]:
# # [문제 0] 프로젝트 목표 정의 (5점)

# 1. 데이터 수집·전처리부터 모델 학습, 성능 평가까지 이어지는 머신러닝 파이프라인의 전체 구조를 명확히 제시하세요.
# 2. 학습(Train) 및 평가(Test) 과정에서 발생할 수 있는 데이터 누수(Data Leakage)를 방지하기 위한 데이터 분리 및 처리 전략을 포함하여 기술하세요.

base_path = r"/Users/kdt_0900_lyn/data_analysis/resource/dataset"
file_name = r"titanic.csv"
file_path = os.path.join(base_path, file_name) 

df = pd.read_csv(file_path)
df

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C


In [14]:
# 1. 결측치
df = df.drop("Cabin",axis=1)
df
# 2. 인코딩  

# 3. 스케일링

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,S
...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C


In [ ]:
# [문제 1] 데이터 확인 및 EDA (10점)

# 1. 데이터의 행/열 개수, 컬럼명, 결측치 개수(컬럼별) 요약을 작성하고, 이상치를 탐색를 각 컬럼의 이상치로 판단하거나 이상치로 보지 않은 근거를 서술하세요(5점)
# 2. Survived 클래스 비율(생존/사망 비율)을 계산하고 해석하세요. (5점)

 



In [ ]:
# [문제 2] 다음 조건을 만족하도록 전처리를 설계하세요. (20점)

# 1. 수치형/범주형 변수를 구분하라. (5점)
# 2. 결측치 처리 전략을 각각 제시하세요. (예: age, embarked 등) (5점)
# 3. 범주형 인코딩 방법(원-핫 등)과 수치형 스케일링 필요 여부를 판단하고 근거를 서술하세요. (5점)
# 4. 스케일링을 표준화로 할지 정규화로 할지 판단 근거를 서술하세요. (5점)

 





In [ ]:
# [문제 3] 데이터 분리 및 검증 전략 (15점)

# 1. 학습/테스트 데이터를 분리하세요. (7점)
# 2. 분리 시 stratify(계층화) 적용 여부를 결정하고 이유를 설명하세요. (8점)

 




In [ ]:
# [문제 4] 모델 학습 (20점)

# 1. 기본 모델 1개 이상: (예: Logistic Regression, Lasso, Ridge, RandomForest 등) (5점)
# 2. 모델 학습 및 예측까지 이어지는 전체 흐름과 각 단계별 핵심 코드를 작성하세요. (10점)
# 3. 파라미터와 하이퍼파라미터의 차이를 서술하고, 하이퍼파라미터 튜닝을 “최소 1개 파라미터”만이라도 시도하세요.(5점)

 






In [ ]:
# [문제 5] 평가 및 리포트 (20점)

# 1. Accuracy, Precision, Recall, F1 중 최소 2개 이상 사용하여 모델을 평가하세요. (10점)
# 2. Confusion Matrix 해석하세요. (5점)
# 3. 어떤 오류가 더 치명적인지(예: 생존자를 사망으로 예측 vs 반대) 상황을 가정해 서술하세요. (5점)

 




In [ ]:
# [문제 6] 최종 제출물 구성 (10점)

# 1. 데이터 요약(EDA 결과 핵심 3가지)하여 서술하세요. (3점)
# 2. 전처리 전략 요약(결측/인코딩/스케일링)하여 서술하세요. (3점)
# 3. 모델 선택 이유와 학습 설정(튜닝 포함)을 요약하여 서술하세요. (2점)
# 4. 최종 성능 지표 및 해석과 개선 아이디어 1개 이상 서술하세요. (2점)